## Imports and connection

In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/checking-logs.sqlite")

## Schema of test

In [2]:
schema_test = pd.io.sql.read_sql("PRAGMA table_info(test);", con)
schema_test

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## First 10 rows preview

In [3]:
test_head = pd.io.sql.read_sql("SELECT * FROM test LIMIT 10;", con)
test_head

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## Min delta (df_min),(df_max),Avg delta 

In [4]:
q_min = """
SELECT
  t.uid,
  (julianday(t.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0 AS min_diff
FROM test t
JOIN deadlines d ON d.labs = t.labname
WHERE t.labname <> 'project1'
ORDER BY min_diff ASC
LIMIT 1;
"""
df_min = pd.io.sql.read_sql(q_min, con)
df_min

,uid,min_diff
0,user_30,-202.38473


In [13]:
q_max = """
SELECT
  t.uid,
  (julianday(t.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0 AS max_diff
FROM test t
JOIN deadlines d ON d.labs = t.labname
WHERE t.labname <> 'project1'
ORDER BY max_diff DESC
LIMIT 1;
"""
df_max = pd.io.sql.read_sql(q_max, con)
df_max

,uid,max_diff
0,user_25,-2.867236


In [3]:
q_avg = """
SELECT
  AVG((julianday(t.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0) AS avg_diff
FROM test t
JOIN deadlines d ON d.labs = t.labname
WHERE t.labname <> 'project1';
"""
df_avg = pd.io.sql.read_sql(q_avg, con)
df_avg

,avg_diff
0,-89.687686


## views_diff table (one query)

In [6]:
q_views_diff = """
WITH user_diff AS (
  SELECT
    t.uid,
    AVG((julianday(t.first_commit_ts) - julianday(d.deadlines, 'unixepoch')) * 24.0) AS avg_diff
  FROM test t
  JOIN deadlines d ON d.labs = t.labname
  WHERE t.labname <> 'project1'
  GROUP BY t.uid
),
user_views AS (
  SELECT uid, COUNT(*) AS pageviews
  FROM pageviews
  WHERE uid LIKE 'user_%'
  GROUP BY uid
)
SELECT
  ud.uid,
  ud.avg_diff,
  COALESCE(uv.pageviews, 0) AS pageviews
FROM user_diff ud
LEFT JOIN user_views uv ON uv.uid = ud.uid;
"""
views_diff = pd.io.sql.read_sql(q_views_diff, con)
views_diff.head()

,uid,avg_diff,pageviews
0,user_1,-65.119644,28
1,user_10,-75.242310,89
2,user_14,-159.568696,143
3,user_17,-62.207514,47
4,user_18,-6.367907,3


## Correlation (Pandas corr)

In [7]:
corr_value = views_diff["pageviews"].corr(views_diff["avg_diff"])
corr_value

-0.27914309051691

In [ ]:
con.close()